### Data Profiling
Documentation from Databricks is [here](https://www.databricks.com/blog/2023/04/03/pandas-profiling-now-supports-apache-spark.html) and [here as well](https://databrickslabs.github.io/dqx/).

### Databricks DQX library modules (used for this demo)
- **WorkspaceClient** Authenticate the Databricks SDK for Python with your Databricks account or workspace to perform data quality checking with DQX
- **DQProfiler** Profile the data to understand the structure (Datatypes, Cardinality, Unique values, Stats)
- **DQGenerator** To generate data quality check rules based on the data that is profiled
- **DQEngine** Run the data quality checks 




In [0]:
%pip install ydata-profiling

In [0]:
%pip install databricks-labs-dqx

In [0]:
# We dont need to restart if library is already installed
# one way to restart
# dbutils.library.restartPython()

#second method to restart
%restart_python

In [0]:
#Get the file 
dfPetLic = spark\
.read.option("inferSchema","true")\
.option("header","true")\
.option("sep",",")\
.csv("/Volumes/workspace/default/myvolume/Seattle_Pet_Licenses.csv")


In [0]:
#DataFrame describe
dfPetLic.describe()

In [0]:
#DataFrame display
display(dfPetLic)

In [0]:
dbutils.data.summarize(dfPetLic)

In [0]:
#Trim the dataframe
dfPetLic_trimed = dfPetLic.drop("Animal's Name","License Issue Date")
dfPetLic_trimed = dfPetLic_trimed.withColumnRenamed("License Number", "License_Number")
dfPetLic_trimed = dfPetLic_trimed.withColumnRenamed("Primary Breed", "Primary_Breed")
dfPetLic_trimed = dfPetLic_trimed.withColumnRenamed("Secondary Breed", "Secondary_Breed")
dfPetLic_trimed = dfPetLic_trimed.withColumnRenamed("ZIP Code", "ZIP_Code")
display(dfPetLic_trimed)

In [0]:
from ydata_profiling import ProfileReport

df = dfPetLic.toPandas()
profile = ProfileReport(df, title="Profiling Report")


In [0]:
json_data = profile.to_json()
print(json_data)

In [0]:
profile.to_notebook_iframe()

In [0]:
from databricks.labs.dqx.profiler.profiler import DQProfiler          # analyzes data to generate summary statistics
from databricks.labs.dqx.profiler.generator import DQGenerator        # uses generated findings to create automated data quality rules and validation constraints
from databricks.labs.dqx.profiler.dlt_generator import DQDltGenerator # used to generated native Delta Live Tables (DLT) expectations
from databricks.labs.dqx.engine import DQEngine                       # executes these rules and splits data into clean / quarantined sets
from databricks.sdk import WorkspaceClient

'''
----------------------------------------------------------------
DQX Profiler                      | DQX Generator
----------------------------------------------------------------
Analyzes data structure & content | Creates suggests data quality
Summary Statistics                | rules.
                                  |
Input is Raw DataFrame / Table    | Profiler Output
----------------------------------------------------------------
'''

In [0]:
import json

profile_options = {
    "round": True,           # round the min/max values
    "max_in_count": 10,      # generate is_in if we have less than x percent of distinct values
    "distinct_ratio": 0.05,  # generate is_distinct if we have less than 1 percent of distinct values
    "max_null_ratio": 0.01,  # generate is_null if we have less than 1 percent of nulls
    "remove_outliers": True, # remove outliers (True implies num_sigmas is active)
    "outlier_columns": [],   # remove outliers in the columns
    "num_sigmas": 3,         # If mean value is +/- 3(num_sigmas value) times Std.Deviation => acceptable range. Anything outside is outliers
    "trim_strings": True,    # trim whitespace from strings
    "max_empty_ratio": 0.01, # generate is_empty if we have less than 1 percent of empty strings
    "sample_fraction": 0.3,  # value between 0.0 and 1.0. (0.3 = 30%). fraction of data to be used for profiling
    "sample_seed": None,     # seed for sampling
    "limit": 10000,          # limit the number of samples (Caps total volume of data to be profiled)
}

# if only specific columns to profile 
columns_to_profile = ["Species", "Primary_Breed", "Secondary_Breed","License_Number", "ZIP_Code"] 

# The engine requires a Databricks workspace client for authentication and interaction with the Databricks workspace
ws = WorkspaceClient()

#for c in ws.clusters.list():
#  print(c.cluster_name)
#print("-----------------------------------------------------------------------------------------------")
#db_fs = ws.dbutils.fs.ls('/')
#for f in db_fs:
#  print(f.path)
#print("-----------------------------------------------------------------------------------------------")

# Prifile the data
profiler = DQProfiler(ws)

#ways to profile data
#summary_stats, profiles = profiler.profile(dfPetLic) # without considering profile options
summary_stats, profiles = profiler.profile(dfPetLic_trimed,options=profile_options, columns=columns_to_profile)

# print the data profile and stats of the data
#Notice that Species has less number of unique values (low cardinality) hence it identifies as is_in
# https://databrickslabs.github.io/dqx/docs/reference/quality_rules/#row-level-quality-checks
#print(profiles)  # Based on the input data a profile is created.
for pf in profiles: # print list
  print(pf)  

print("-----------------------------------------------------------------------------------------------")
#print(summary_stats)
# for key, value in summary_stats.items(): #print dictionary
#    print (key,value)

# try printing in a formated way 
json_formatted = json.dumps(summary_stats, indent=4)
print(json_formatted)
print("-----------------------------------------------------------------------------------------------")
'''
Note: In summary stats you see 25%, 50%, 75% -> provides info aboutn approximate percentils of NUMBER field data. Helps to identify outliers
Example: Lets say if we have a column as customer_age
         and if profile returns values like "25%": 29, "50%": 42, and "75%": 64, it means:
             25% of values are <= 29.
             50% of values are <= 42.
             75% of values are <= 64. 
Another example: Lets say if we have a column as emp_salary
         and if profile returns values like "25%": 10,000, "50%": 20,000, and "75%": 30,000, it means:
             25% of employees earn <= 10,000.
             50% of employees earn <= 20,000.
             75% of employees earn <= 30,000,
'''


In [0]:
# generate DQX quality rules based on the profile
generator = DQGenerator(ws)
checks = generator.generate_dq_rules(profiles)
for chk in checks:
    print(chk)
print("-----------------------------------------------------------------------------------------------")

In [0]:
# drop few columns from DF
# apply the checks generated based on profile to validate
dqengine = DQEngine(ws)
results = dqengine.apply_checks_by_metadata(dfPetLic_trimed, checks)
display(results)    

In [0]:
#Create a user defined check that is not part of auto generated check constraints
import yaml
udChecks = yaml.safe_load("""
- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Species in ('Dog','Cat')"
      msg: "Invalid Species name"
      
""")

dqengine = DQEngine(ws)
valid,quarantine = dqengine.apply_checks_by_metadata_and_split(dfPetLic_trimed, udChecks,globals())
display(quarantine)


In [0]:
quarantine.select("*").show()

In [0]:
aggresult = quarantine.groupBy("Species").count()
display(aggresult)

In [0]:
quarantine.createOrReplaceTempView("Invalid_Species")

# SQL
vsql ="select Species, count(*) as cnt from Invalid_Species " + \
"group by Species limit 10"
spark.sql(vsql).show()

In [0]:
# get counts for all not null values
from pyspark.sql.functions import col, when, count
quarantine_Columns=["Species","Secondary_Breed"]
quarantine.select([count(when(col(c).isNotNull(), c)).alias(c) for c in quarantine_Columns]
   ).show()

In [0]:
# get counts of all null values
quarantine.select([count(when(col(c).isNull(), c)).alias(c) for c in quarantine_Columns]
   ).show()

In [0]:
species_dist = dfPetLic.select("Species").distinct()
species_dist.show()
print(species_dist.count())